# Step 2 — Early-Warning Model at Full Scale (Open TG-GATEs)

Step 1 produced **396 pre-injury compound–dose arms across 139 compounds**, 48 of them
positive (12.1%). This notebook builds the forward-prediction model on those arms and
runs the validation audit at full scale.

### The join is verified
The Zenodo matrix has 6,765 sample columns; Step 1's `array_labels_union.csv` has
exactly 6,765 rows. Column names are TG-GATEs BARCODEs with leading zeros, so
`zfill(12)` matches them one to one. Nothing is lost in the join.

### What this answers
1. Does the day-4 transcriptome, measured in histopathologically normal liver, predict
   confirmed injury at day 15/29 in a compound the model has never seen?
2. Does it beat **dose alone**? On this pre-injury arm set, dose-only AUC is **0.690** —
   far below the 0.893 it reached on the conventional detection task, because excluding
   arms already injured at day 4 removes most high-dose positives. That is the number to
   beat.
3. How much optimism does the conventional random-split design introduce?

### Evaluation unit
The label is a property of the **arm**, not of the individual rat. Samples within an arm
are replicates sharing one outcome, so sample-level predictions are averaged to an arm
score and evaluation is done on **396 arms**, not 1,188 samples. Scoring at sample level
would treat replicates as independent and overstate precision.

### Prerequisites
Upload `step1_outputs.zip` from Step 1b (or re-run it). Optionally upload your own
`expression_matrix_gene1.csv` for the provenance check in section 4.

## 1. Setup

In [ ]:
!pip install xgboost shap mygene -q

import json, re, sys, gc
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import ttest_ind
from statsmodels.stats.multitest import multipletests
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneGroupOut, StratifiedKFold, train_test_split
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             balanced_accuracy_score, matthews_corrcoef)
from xgboost import XGBClassifier

RANDOM_STATE = 42
EARLY_DAY    = 4
N_FEATURES   = 100
RULE         = "union"          # primary label rule; others run as sensitivity

WORK = Path("/content/step2"); WORK.mkdir(parents=True, exist_ok=True)
STEP1 = Path("/content/tggates_annot")      # where Step 1b wrote its CSVs
MATRIX = WORK / "affy_tggates_repeat.tsv"
ZEN = "https://zenodo.org/records/20437047/files/"

np.random.seed(RANDOM_STATE)
print("work:", WORK)

## 2. Load the Step 1 arm labels

If `/content/tggates_annot` is empty, upload `step1_outputs.zip` and unzip it there.

In [ ]:
if not (STEP1 / f"arms_{RULE}.csv").exists():
    print("Step 1 outputs not found. Upload step1_outputs.zip now.")
    from google.colab import files
    up = files.upload()
    import zipfile
    STEP1.mkdir(parents=True, exist_ok=True)
    for name in up:
        with zipfile.ZipFile(name) as z:
            z.extractall(STEP1)
    print("extracted to", STEP1)

arms = pd.read_csv(STEP1 / f"arms_{RULE}.csv")
labels = pd.read_csv(STEP1 / f"array_labels_{RULE}.csv")
labels["bc"] = labels.BARCODE.astype(str).str.zfill(12)

admitted = arms[arms.status == "admitted"].copy()
print(f"admitted arms: {len(admitted)}  positive: {int(admitted.outcome.sum())} "
      f"({100*admitted.outcome.mean():.1f}%)  compounds: {admitted.compound.nunique()}")

# day-4 arrays belonging to admitted arms -> these are the model inputs
adm_keys = set(zip(admitted.compound, admitted.dose_level))
day4 = labels[(labels.day == EARLY_DAY)].copy()
day4["is_admitted"] = [ (c, d) in adm_keys
                        for c, d in zip(day4.COMPOUND_NAME, day4.DOSE_LEVEL) ]
model_samples = day4[day4.is_admitted].copy()

outcome_map = {(c, d): int(o) for c, d, o in
               zip(admitted.compound, admitted.dose_level, admitted.outcome)}
model_samples["arm"] = (model_samples.COMPOUND_NAME + " | " + model_samples.DOSE_LEVEL)
model_samples["y"] = [outcome_map[(c, d)] for c, d
                      in zip(model_samples.COMPOUND_NAME, model_samples.DOSE_LEVEL)]

print(f"day-4 arrays in admitted arms: {len(model_samples)}")
print(f"  arms represented: {model_samples.arm.nunique()}")
print(f"  compounds        : {model_samples.COMPOUND_NAME.nunique()}")
print(f"  positive samples : {int(model_samples.y.sum())}")

# day-4 controls are loaded too, for reference statistics (not used as model rows)
ctrl4 = day4[day4.DOSE_LEVEL == "Control"].copy()
NEEDED = sorted(set(model_samples.bc) | set(ctrl4.bc))
print(f"\ncolumns to load from the matrix: {len(NEEDED)} "
      f"({len(model_samples)} treated + {len(ctrl4)} control)")

## 3. Download the expression matrix

666 MB. `-c` resumes a broken transfer instead of restarting it.

In [ ]:
import subprocess

URL = ZEN + "affy_tggates_repeat.tsv?download=1"
if not MATRIX.exists() or MATRIX.stat().st_size < 600_000_000:
    print("downloading ~666 MB (resumable) ...")
    subprocess.run(["wget", "-c", "-q", "--show-progress", "-O", str(MATRIX), URL],
                   check=True)
else:
    print("already downloaded")
print(f"{MATRIX.name}: {MATRIX.stat().st_size/1e6:.0f} MB")

## 4. Selective load

The file is genes × samples, so only the needed sample columns are parsed. Loading all
6,765 columns would cost roughly 8× the memory for data that is never used.

In [ ]:
header = pd.read_csv(MATRIX, sep="\t", nrows=0)
all_cols = list(header.columns)
idx_col = all_cols[0]
present = [c for c in NEEDED if c in set(all_cols)]
missing = [c for c in NEEDED if c not in set(all_cols)]
print(f"matrix columns: {len(all_cols)-1} samples")
print(f"requested {len(NEEDED)} | found {len(present)} | missing {len(missing)}")
if missing[:5]:
    print("  first missing:", missing[:5])

expr = pd.read_csv(MATRIX, sep="\t", usecols=[idx_col] + present,
                   index_col=0, dtype={c: "float32" for c in present})
expr.index = expr.index.astype(str)
print(f"\nloaded: {expr.shape[0]} genes x {expr.shape[1]} samples "
      f"({expr.memory_usage(deep=True).sum()/1e6:.0f} MB)")
X_all = expr.T                      # samples x genes
del expr; gc.collect()
print("transposed:", X_all.shape)

## 5. Provenance check (optional but recommended)

This is a third-party deposit. If you upload your own `expression_matrix_gene1.csv`,
this compares the two pipelines on shared samples.

The two use different CDFs — yours maps Affymetrix probesets to symbols, Zenodo uses
BrainArray ENTREZG — so values will not be identical. The meaningful test is **identity**:
for a shared barcode, correlation with *itself* across pipelines should clearly exceed
correlation with *other* samples. If it does not, the deposit's sample labelling is
unreliable and the CEL route is safer.

In [ ]:
OWN = Path("/content/expression_matrix_gene1.csv")
if OWN.exists():
    own = pd.read_csv(OWN, index_col=0)
    own = own.dropna(subset=["GeneSymbol"])
    cels = [c for c in own.columns if c.endswith(".CEL")]
    own_g = own.groupby("GeneSymbol")[cels].mean().T
    own_g.index = [re.sub(r"\.CEL$", "", i) for i in own_g.index]

    shared = [b for b in own_g.index if b in X_all.index]
    print(f"shared barcodes: {len(shared)}")
    if len(shared) >= 6:
        test = shared[:15]
        # The two pipelines use different gene spaces, so values are not comparable
        # directly. What IS comparable is the between-sample similarity structure:
        # if both label samples the same way, the sample-sample correlation matrices
        # should agree.
        ca = np.corrcoef(X_all.loc[test].values.astype(float))
        cb = np.corrcoef(own_g.loc[test].values.astype(float))
        tri = np.triu_indices(len(test), 1)
        agree = np.corrcoef(ca[tri], cb[tri])[0, 1]
        print(f"similarity-structure agreement over {len(test)} shared samples: "
              f"r = {agree:.3f}")
        print("Expect > 0.8. A low value means the two pipelines disagree about")
        print("which sample is which, and the raw CEL route should be used instead.")
    else:
        print("too few shared barcodes to check")
else:
    print("own matrix not uploaded - skipping. Upload expression_matrix_gene1.csv to run it.")

## 6. Map Entrez IDs to gene symbols

In [ ]:
genes = [g for g in X_all.columns if not str(g).startswith("AFFX")]
X_all = X_all[genes]
print("genes after dropping AFFX controls:", X_all.shape[1])

sym = {}
try:
    import mygene
    mg = mygene.MyGeneInfo()
    res = mg.querymany(list(X_all.columns), scopes="entrezgene",
                       fields="symbol", species="rat", verbose=False)
    sym = {str(r["query"]): r.get("symbol") for r in res if not r.get("notfound")}
    print(f"mapped {len(sym)} / {X_all.shape[1]} Entrez IDs to symbols")
except Exception as e:
    print("mygene unavailable:", e, "- continuing with Entrez IDs")

ENTREZ_TO_SYMBOL = sym
def label_gene(g):
    return ENTREZ_TO_SYMBOL.get(str(g), str(g))

## 7. Assemble the modelling dataset

`X` is day-4 expression, `y` the arm's later outcome, `groups` the compound (the LOCO
unit), and `arm` the evaluation unit.

In [ ]:
ms = model_samples[model_samples.bc.isin(X_all.index)].copy()
X = X_all.loc[ms.bc.values]
y = ms.y.values
groups = ms.COMPOUND_NAME.values
arm_id = ms.arm.values

print(f"X: {X.shape}  positive samples: {y.sum()} ({100*y.mean():.1f}%)")
print(f"compounds: {len(set(groups))}  arms: {len(set(arm_id))}")

arm_truth = (pd.DataFrame({"arm": arm_id, "y": y, "compound": groups})
               .groupby("arm").agg(y=("y", "max"), compound=("compound", "first")))
print(f"\narm-level: {len(arm_truth)} arms, {int(arm_truth.y.sum())} positive "
      f"({100*arm_truth.y.mean():.1f}%)")

dose_ord = ms.DOSE_LEVEL.map({"Low": 1, "Middle": 2, "High": 3}).values

## 8. Pipeline pieces (fitted inside each fold only)

In [ ]:
def differential_genes(Xtr, ytr, alpha=0.05, min_lfc=0.5):
    pos, neg = Xtr.values[ytr == 1], Xtr.values[ytr == 0]
    if len(pos) < 2 or len(neg) < 2:
        return Xtr.columns[:500]
    _, p = ttest_ind(pos, neg, axis=0, equal_var=False)
    lfc = pos.mean(0) - neg.mean(0)
    ok = ~np.isnan(p); padj = np.ones_like(p)
    padj[ok] = multipletests(p[ok], method="fdr_bh")[1]
    keep = (padj < alpha) & (np.abs(lfc) >= min_lfc)
    return Xtr.columns[keep] if keep.sum() >= 20 else Xtr.columns[np.argsort(p)[:500]]

def select_features(Xtr, ytr, n=N_FEATURES):
    rf = RandomForestClassifier(n_estimators=600, class_weight="balanced_subsample",
                                random_state=RANDOM_STATE, n_jobs=-1).fit(Xtr, ytr)
    return Xtr.columns[np.argsort(rf.feature_importances_)[::-1][:n]]

def make_model(ytr):
    pos = max(int(ytr.sum()), 1)
    return XGBClassifier(n_estimators=400, max_depth=3, learning_rate=0.05,
                         subsample=0.8, colsample_bytree=0.8,
                         scale_pos_weight=(len(ytr) - pos) / pos,
                         random_state=RANDOM_STATE, eval_metric="logloss")

def fit_predict(Xtr, ytr, Xte):
    deg = differential_genes(Xtr, ytr)
    feats = select_features(Xtr[deg], ytr)
    model = make_model(ytr).fit(Xtr[feats], ytr)
    return model.predict_proba(Xte[feats])[:, 1], feats, model

def fit_predict_fast(Xtr, ytr, Xte, n=40):
    """Cheaper pipeline for the permutation test. The observed statistic is
    recomputed with THIS pipeline too, so observed and null stay comparable --
    a null built with a different pipeline would not be a valid reference."""
    _, p = ttest_ind(Xtr.values[ytr == 1], Xtr.values[ytr == 0], axis=0, equal_var=False)
    p = np.nan_to_num(p, nan=1.0)
    feats = Xtr.columns[np.argsort(p)[:n]]
    m = XGBClassifier(n_estimators=120, max_depth=3, learning_rate=0.1,
                      scale_pos_weight=(len(ytr) - max(int(ytr.sum()), 1))
                                        / max(int(ytr.sum()), 1),
                      random_state=RANDOM_STATE, eval_metric="logloss")
    return m.fit(Xtr[feats], ytr).predict_proba(Xte[feats])[:, 1]

def arm_scores(sample_proba, arms_vec):
    """Average replicate predictions within an arm -- the evaluation unit."""
    return (pd.DataFrame({"arm": arms_vec, "p": sample_proba})
              .groupby("arm")["p"].mean())

def evaluate(truth, score):
    j = truth.join(score.rename("p"), how="inner").dropna()
    if j.y.nunique() < 2:
        return {"n_arms": len(j), "auc": None, "ap": None, "bacc": None, "mcc": None}
    pred = (j.p >= j.p.median()).astype(int)
    return {"n_arms": int(len(j)), "n_pos": int(j.y.sum()),
            "auc": float(roc_auc_score(j.y, j.p)),
            "ap": float(average_precision_score(j.y, j.p)),
            "bacc": float(balanced_accuracy_score(j.y, pred)),
            "mcc": float(matthews_corrcoef(j.y, pred))}

def bootstrap_ci(truth, score, n=2000):
    j = truth.join(score.rename("p"), how="inner").dropna()
    rng = np.random.default_rng(RANDOM_STATE); out = []
    for _ in range(n):
        i = rng.integers(0, len(j), len(j))
        s = j.iloc[i]
        if s.y.nunique() > 1:
            out.append(roc_auc_score(s.y, s.p))
    return [float(np.percentile(out, 2.5)), float(np.percentile(out, 97.5))]

## 9. The audit — three evaluations side by side

**B** is the honest estimate. **A** shows how much optimism a random split adds. **C** is
the bar the transcriptome must clear: dose alone, which scores 0.690 on these arms.

In [ ]:
RESULTS = {}

# --- B. leave-one-compound-out (primary) ---
print("B. leave-one-compound-out ...")
oof = np.zeros(len(y)); fold_feats = {}
for k, (tr, te) in enumerate(LeaveOneGroupOut().split(X, y, groups)):
    if len(np.unique(y[tr])) < 2:
        oof[te] = np.nan; continue
    oof[te], feats, _ = fit_predict(X.iloc[tr], y[tr], X.iloc[te])
    fold_feats[groups[te][0]] = list(feats)
    if (k + 1) % 20 == 0:
        print(f"   {k+1} folds")
ok = ~np.isnan(oof)
sc_loco = arm_scores(oof[ok], arm_id[ok])
RESULTS["B_loco"] = evaluate(arm_truth, sc_loco)
RESULTS["B_loco"]["auc_ci95"] = bootstrap_ci(arm_truth, sc_loco)
print("   ", RESULTS["B_loco"])

# --- A. random stratified split, the conventional design ---
print("\nA. random stratified split ...")
tr, te = train_test_split(np.arange(len(y)), test_size=0.25,
                          stratify=y, random_state=RANDOM_STATE)
p_a, _, _ = fit_predict(X.iloc[tr], y[tr], X.iloc[te])
RESULTS["A_random_split"] = evaluate(arm_truth, arm_scores(p_a, arm_id[te]))
print("   ", RESULTS["A_random_split"])

# --- C. dose-only baseline, same LOCO scheme ---
print("\nC. dose-only baseline ...")
oof_d = np.zeros(len(y))
D = dose_ord.reshape(-1, 1)
for tr, te in LeaveOneGroupOut().split(D, y, groups):
    if len(np.unique(y[tr])) < 2:
        oof_d[te] = np.nan; continue
    oof_d[te] = make_model(y[tr]).fit(D[tr], y[tr]).predict_proba(D[te])[:, 1]
okd = ~np.isnan(oof_d)
RESULTS["C_dose_loco"] = evaluate(arm_truth, arm_scores(oof_d[okd], arm_id[okd]))
RESULTS["C_dose_raw"] = {"auc": float(roc_auc_score(
    arm_truth.y, arm_truth.index.map(
        pd.Series(dose_ord, index=arm_id).groupby(level=0).first())))}
print("   ", RESULTS["C_dose_loco"])
print("    raw dose:", RESULTS["C_dose_raw"])

## 10. Per-compound breakdown and permutation test

In [ ]:
per_cmp = {}
dfp = pd.DataFrame({"arm": arm_id[ok], "p": oof[ok], "compound": groups[ok]})
agg = dfp.groupby("arm").agg(p=("p", "mean"), compound=("compound", "first"))
j = arm_truth.join(agg[["p"]], how="inner")
for c, g in j.groupby("compound"):
    if g.y.nunique() > 1:
        per_cmp[c] = {"n_arms": len(g), "n_pos": int(g.y.sum()),
                      "auc": float(roc_auc_score(g.y, g.p))}
RESULTS["per_compound"] = per_cmp
print(f"compounds with both classes (AUC computable): {len(per_cmp)}")
if per_cmp:
    v = [d["auc"] for d in per_cmp.values()]
    print(f"  median {np.median(v):.3f}  range {min(v):.3f}-{max(v):.3f}")

# Permutation test. A full-pipeline null would mean NP x 139 model fits and is
# not practical here, so both the observed statistic and the null are computed
# with the lighter pipeline. Set NP higher if you have the compute.
NP = 10
print(f"permutation test, light pipeline, {NP} permutations ...")

def loco_light(labels_vec):
    o = np.full(len(labels_vec), np.nan)
    for tr, te in LeaveOneGroupOut().split(X, labels_vec, groups):
        if len(np.unique(labels_vec[tr])) < 2:
            continue
        o[te] = fit_predict_fast(X.iloc[tr], labels_vec[tr], X.iloc[te])
    k = ~np.isnan(o)
    sc = arm_scores(o[k], arm_id[k])
    truth = (pd.DataFrame({"arm": arm_id, "y": labels_vec})
               .groupby("arm").y.max().to_frame())
    j = truth.join(sc.rename("p"), how="inner").dropna()
    return roc_auc_score(j.y, j.p) if j.y.nunique() > 1 else np.nan

obs_light = loco_light(y)
print(f"   observed (light pipeline): {obs_light:.3f}")

at = arm_truth.copy(); rng = np.random.default_rng(RANDOM_STATE); null = []
for i in range(NP):
    shuf = dict(zip(at.index, rng.permutation(at.y.values)))
    a = loco_light(np.array([shuf[x] for x in arm_id]))
    if not np.isnan(a):
        null.append(a)
        print(f"   perm {i+1}/{NP}: {a:.3f}")
RESULTS["permutation"] = {
    "observed_full_pipeline": RESULTS["B_loco"]["auc"],
    "observed_light_pipeline": float(obs_light),
    "null_mean": float(np.mean(null)), "null_sd": float(np.std(null)),
    "n_permutations": len(null),
    "p": float((np.sum(np.array(null) >= obs_light) + 1) / (len(null) + 1))}
print("   ", RESULTS["permutation"])

## 11. Which genes carry the anticipatory signal

Attribution is computed on held-out compounds, and genes are reported with the number of
folds that selected them. A gene chosen in one fold may reflect that partition rather
than a reproducible early response.

In [ ]:
from collections import Counter
cnt = Counter(g for feats in fold_feats.values() for g in feats)
stab = (pd.DataFrame({"gene": list(cnt), "folds": list(cnt.values())})
          .assign(symbol=lambda d: d.gene.map(label_gene),
                  pct=lambda d: (100 * d.folds / max(len(fold_feats), 1)).round(1))
          .sort_values("folds", ascending=False))
print(f"folds: {len(fold_feats)} | genes ever selected: {len(stab)}")
print("\nmost consistently selected:")
print(stab.head(30)[["symbol", "gene", "folds", "pct"]].to_string(index=False))
stab.to_csv(WORK / "feature_stability.csv", index=False)

import shap
core = stab[stab.pct >= 50].gene.tolist()[:N_FEATURES]
if len(core) >= 10:
    m = make_model(y).fit(X[core], y)
    sv = shap.TreeExplainer(m).shap_values(X[core])
    imp = (pd.DataFrame({"gene": core, "mean_abs_shap": np.abs(sv).mean(0)})
             .assign(symbol=lambda d: d.gene.map(label_gene))
             .sort_values("mean_abs_shap", ascending=False))
    print("\ntop SHAP contributors among stable genes:")
    print(imp.head(20)[["symbol", "gene", "mean_abs_shap"]].to_string(index=False))
    imp.to_csv(WORK / "shap_stable_genes.csv", index=False)
else:
    print(f"\nonly {len(core)} genes reach 50% fold selection - too unstable for SHAP")

## 12. Sensitivity across label rules

In [ ]:
sens = {}
for rule in ["union", "any_finding", "moderate_plus", "injury"]:
    f = STEP1 / f"arms_{rule}.csv"
    if not f.exists():
        continue
    ar = pd.read_csv(f); ad = ar[ar.status == "admitted"]
    om = {(c, d): int(o) for c, d, o in zip(ad.compound, ad.dose_level, ad.outcome)}
    lb = pd.read_csv(STEP1 / f"array_labels_{rule}.csv")
    lb["bc"] = lb.BARCODE.astype(str).str.zfill(12)
    s = lb[(lb.day == EARLY_DAY)].copy()
    s = s[[(c, d) in om for c, d in zip(s.COMPOUND_NAME, s.DOSE_LEVEL)]]
    s = s[s.bc.isin(X_all.index)]
    if s.empty or len(s.COMPOUND_NAME.unique()) < 3:
        continue
    yy = np.array([om[(c, d)] for c, d in zip(s.COMPOUND_NAME, s.DOSE_LEVEL)])
    if len(np.unique(yy)) < 2:
        continue
    XX = X_all.loc[s.bc.values][X.columns]
    gg = s.COMPOUND_NAME.values
    aa = (s.COMPOUND_NAME + " | " + s.DOSE_LEVEL).values
    o = np.full(len(yy), np.nan)
    for tr, te in LeaveOneGroupOut().split(XX, yy, gg):
        if len(np.unique(yy[tr])) < 2:
            continue
        o[te], _, _ = fit_predict(XX.iloc[tr], yy[tr], XX.iloc[te])
    k = ~np.isnan(o)
    t = pd.DataFrame({"arm": aa, "y": yy}).groupby("arm").y.max().to_frame()
    sens[rule] = evaluate(t, arm_scores(o[k], aa[k]))
    print(f"{rule:15s} {sens[rule]}")
RESULTS["sensitivity"] = sens

## 13. Summary and save

In [ ]:
rows = [
    ("A  random split (conventional)", RESULTS["A_random_split"].get("auc")),
    ("B  leave-one-compound-out",      RESULTS["B_loco"].get("auc")),
    ("C  dose only, LOCO",             RESULTS["C_dose_loco"].get("auc")),
    ("C  dose only, raw",              RESULTS["C_dose_raw"].get("auc")),
]
tbl = pd.DataFrame(rows, columns=["evaluation", "arm_level_auc"])
print(tbl.to_string(index=False))
print(f"\nLOCO 95% CI: {RESULTS['B_loco'].get('auc_ci95')}")
print(f"permutation p: {RESULTS['permutation']['p']:.3f}")

b = RESULTS["B_loco"].get("auc"); d = RESULTS["C_dose_loco"].get("auc")
if b is not None and d is not None:
    print(f"\ntranscriptome minus dose: {b - d:+.3f}")
    print("A positive gap is the finding. A gap at or below zero means expression")
    print("adds nothing beyond exposure level -- report that outcome as it stands.")

tbl.to_csv(WORK / "step2_summary_table.csv", index=False)
(WORK / "step2_results.json").write_text(json.dumps(RESULTS, indent=2, default=str))

import shutil
arch = shutil.make_archive("/content/step2_outputs", "zip", WORK)
print("\narchive:", arch)
try:
    from google.colab import files; files.download(arch)
except Exception as e:
    print("files are in", WORK, "|", e)

## Reading the result

The comparison that matters is **B minus C**: the day-4 transcriptome against dose alone,
both under compound-held-out validation, both scored on arms.

- **B clearly above C** — pre-injury expression carries information about later injury
  beyond exposure level. That is the paper's finding, and the stable gene list in
  section 11 is its biological content.
- **B close to or below C** — expression adds nothing once dose is accounted for. This is
  still a result worth publishing, and a more useful one than the field's current
  position: it bounds what toxicogenomic early warning can deliver under these exposure
  conditions. Report it plainly rather than retreating to the detection task, where high
  numbers come easily and mean little.

Either way, **A minus B** quantifies how much the conventional random-split design
inflates reported performance — a methodological result that stands on its own.

Interpret genes only from the stability table. With 48 positive arms, a gene selected in
a minority of folds is noise.